# Tutorial 4: Cell Type Annotation with Fine-Tuning

Welcome to Tutorial 4! This covers **Phase 5: Cell Type Annotation**.

## Learning Objectives

By the end of this tutorial, you will understand:
1. How to fine-tune a pretrained model for classification
2. Different fine-tuning strategies (full vs frozen encoder)
3. How to evaluate classification performance
4. How to predict cell types on unlabeled data
5. How to interpret model predictions and confidence scores

## Prerequisites

Complete Tutorial 2 or have a pretrained model checkpoint ready.

## Part 1: Understanding Transfer Learning for scRNA-seq

**Transfer learning** allows us to leverage pretrained knowledge:

```
Phase 1: Pretraining (Tutorial 2)
├── Task: Masked Language Modeling
├── Data: All cells (unlabeled)
└── Goal: Learn gene relationships

Phase 2: Fine-tuning (This Tutorial)
├── Task: Cell Type Classification
├── Data: Labeled cells only
└── Goal: Predict cell types
```

### Two Fine-Tuning Strategies:

1. **Frozen encoder**: Only train classification head (faster, less data needed)
2. **Full fine-tuning**: Train entire model (better performance, more data needed)

Let's compare both!

In [ ]:
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## Part 2: Preparing Labeled Data

We'll use PBMC 3k with annotated cell types:

In [ ]:
from scgpt_mini.data import preprocess_adata
from scgpt_mini.tokenizer import GeneVocab

# Load preprocessed PBMC data with labels
adata = sc.datasets.pbmc3k_processed()

print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"\nCell type distribution:")
print(adata.obs['louvain'].value_counts())

In [ ]:
# Preprocess to match pretraining
adata = preprocess_adata(
    adata,
    filter_gene_by_counts=10,
    filter_cell_by_genes=200,
    normalize_total_target=1e4,
    log1p=True,
    subset_hvg=500,
    binning=False,
    inplace=False,
)

print(f"\nPreprocessed: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
# Create label encoder
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
cell_type_labels = label_encoder.fit_transform(adata.obs['louvain'])
n_classes = len(label_encoder.classes_)

print(f"Number of cell types: {n_classes}")
print(f"Cell types: {label_encoder.classes_.tolist()}")

# Add numeric labels to adata
adata.obs['cell_type_id'] = cell_type_labels

## Part 3: Creating Train/Val/Test Splits

Important: Use stratified splits to maintain class balance!

In [ ]:
# Split into train/val/test
train_val_indices, test_indices = train_test_split(
    range(adata.n_obs),
    test_size=0.2,
    stratify=cell_type_labels,
    random_state=42,
)

train_labels = cell_type_labels[train_val_indices]
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.25,  # 0.25 of 0.8 = 0.2 overall
    stratify=train_labels,
    random_state=42,
)

print(f"Train: {len(train_indices)} cells ({len(train_indices)/adata.n_obs:.1%})")
print(f"Val: {len(val_indices)} cells ({len(val_indices)/adata.n_obs:.1%})")
print(f"Test: {len(test_indices)} cells ({len(test_indices)/adata.n_obs:.1%})")

## Part 4: Preparing Data for Fine-Tuning

In [ ]:
from scgpt_mini.tokenizer import tokenize_batch
from scgpt_mini.data import create_dataloader

# Create vocabulary
vocab = GeneVocab(adata.var_names.tolist())

# Tokenize all data
data_matrix = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
gene_names = adata.var_names.values

tokenized_data = tokenize_batch(
    data=data_matrix,
    gene_names=gene_names,
    vocab=vocab,
    append_cls=True,
    include_zero_genes=False,
    return_pt=True,
)

print(f"Tokenized {len(tokenized_data)} cells")

In [ ]:
# Create datasets with labels
train_data = [(tokenized_data[i], cell_type_labels[i]) for i in train_indices]
val_data = [(tokenized_data[i], cell_type_labels[i]) for i in val_indices]
test_data = [(tokenized_data[i], cell_type_labels[i]) for i in test_indices]

# Create dataloaders (no masking for classification)
train_loader = create_dataloader(
    tokenized_data=[x[0] for x in train_data],
    vocab=vocab,
    batch_size=32,
    max_len=1001,
    shuffle=True,
    apply_masking=False,  # No masking for classification
    labels=[x[1] for x in train_data],
)

val_loader = create_dataloader(
    tokenized_data=[x[0] for x in val_data],
    vocab=vocab,
    batch_size=32,
    max_len=1001,
    shuffle=False,
    apply_masking=False,
    labels=[x[1] for x in val_data],
)

test_loader = create_dataloader(
    tokenized_data=[x[0] for x in test_data],
    vocab=vocab,
    batch_size=32,
    max_len=1001,
    shuffle=False,
    apply_masking=False,
    labels=[x[1] for x in test_data],
)

print(f"Created dataloaders with {len(train_loader)} train batches")

## Part 5: Loading Pretrained Model and Adding Classification Head

In [ ]:
from scgpt_mini.model import TransformerModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model configuration (with classification head)
model_config = {
    "vocab_size": len(vocab),
    "d_model": 32,
    "nhead": 2,
    "num_layers": 2,
    "d_hid": 64,
    "dropout": 0.1,
    "max_seq_len": 1001,
    "value_mode": "continuous",
    "n_classes": n_classes,  # Add classification head
}

model = TransformerModel(**model_config, vocab=vocab)

# Load pretrained weights (if available)
checkpoint_path = "./tutorial_output/final_model.pt"
if Path(checkpoint_path).exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load only the encoder weights (classification head is new)
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in checkpoint['model_state_dict'].items() 
                      if k in model_dict and 'cls_decoder' not in k}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    
    print(f"✅ Loaded pretrained encoder from epoch {checkpoint['epoch']}")
    print(f"   Classification head initialized randomly")
else:
    print("⚠️  No checkpoint found. Training from scratch.")

model = model.to(device)
print(f"Model ready with {n_classes} output classes!")

## Part 6: Fine-Tuning Strategy 1 - Frozen Encoder

In [ ]:
from scgpt_mini.tasks.annotation import freeze_encoder, unfreeze_encoder

# Freeze encoder (only train classification head)
freeze_encoder(model)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
print(f"Training only: {trainable_params/total_params:.1%} of model")

In [ ]:
# Set up training for frozen encoder
optimizer_frozen = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,  # Higher LR for classification head only
    weight_decay=0.01,
)

criterion = torch.nn.CrossEntropyLoss()

print("Setup complete for frozen encoder training")

In [ ]:
# Train with frozen encoder
from scgpt_mini.tasks.annotation import train_classification_epoch, evaluate_classification

history_frozen = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

epochs_frozen = 10
for epoch in range(epochs_frozen):
    # Train
    train_loss, train_acc = train_classification_epoch(
        model, train_loader, optimizer_frozen, criterion, device
    )
    
    # Validate
    val_loss, val_acc = evaluate_classification(
        model, val_loader, criterion, device
    )
    
    history_frozen['train_loss'].append(train_loss)
    history_frozen['train_acc'].append(train_acc)
    history_frozen['val_loss'].append(val_loss)
    history_frozen['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{epochs_frozen}: "
          f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
          f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

print("\n✅ Frozen encoder training complete!")

## Part 7: Fine-Tuning Strategy 2 - Full Fine-Tuning

In [ ]:
# Unfreeze all parameters
unfreeze_encoder(model)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,} (100% of model)")

In [ ]:
# Set up training for full fine-tuning
optimizer_full = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,  # Lower LR for full model
    weight_decay=0.01,
)

print("Setup complete for full fine-tuning")

In [ ]:
# Continue training with full fine-tuning
history_full = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

epochs_full = 10
for epoch in range(epochs_full):
    # Train
    train_loss, train_acc = train_classification_epoch(
        model, train_loader, optimizer_full, criterion, device
    )
    
    # Validate
    val_loss, val_acc = evaluate_classification(
        model, val_loader, criterion, device
    )
    
    history_full['train_loss'].append(train_loss)
    history_full['train_acc'].append(train_acc)
    history_full['val_loss'].append(val_loss)
    history_full['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{epochs_full}: "
          f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
          f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

print("\n✅ Full fine-tuning complete!")

## Part 8: Comparing Training Strategies

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training loss
axes[0, 0].plot(history_frozen['train_loss'], label='Frozen Encoder', marker='o')
axes[0, 0].plot(
    range(epochs_frozen, epochs_frozen + epochs_full),
    history_full['train_loss'],
    label='Full Fine-Tuning',
    marker='s',
)
axes[0, 0].axvline(epochs_frozen - 0.5, color='gray', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Validation loss
axes[0, 1].plot(history_frozen['val_loss'], label='Frozen Encoder', marker='o')
axes[0, 1].plot(
    range(epochs_frozen, epochs_frozen + epochs_full),
    history_full['val_loss'],
    label='Full Fine-Tuning',
    marker='s',
)
axes[0, 1].axvline(epochs_frozen - 0.5, color='gray', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Validation Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Training accuracy
axes[1, 0].plot(history_frozen['train_acc'], label='Frozen Encoder', marker='o')
axes[1, 0].plot(
    range(epochs_frozen, epochs_frozen + epochs_full),
    history_full['train_acc'],
    label='Full Fine-Tuning',
    marker='s',
)
axes[1, 0].axvline(epochs_frozen - 0.5, color='gray', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Training Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Validation accuracy
axes[1, 1].plot(history_frozen['val_acc'], label='Frozen Encoder', marker='o')
axes[1, 1].plot(
    range(epochs_frozen, epochs_frozen + epochs_full),
    history_full['val_acc'],
    label='Full Fine-Tuning',
    marker='s',
)
axes[1, 1].axvline(epochs_frozen - 0.5, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Validation Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('finetuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 9: Evaluating on Test Set

In [ ]:
from scgpt_mini.tasks.annotation import predict_cell_types

# Get predictions on test set
predictions, confidences = predict_cell_types(
    model=model,
    dataloader=test_loader,
    device=device,
)

# Get true labels
true_labels = np.array([cell_type_labels[i] for i in test_indices])

# Compute metrics
accuracy = accuracy_score(true_labels, predictions)
balanced_acc = balanced_accuracy_score(true_labels, predictions)

print(f"Test Set Performance:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Balanced Accuracy: {balanced_acc:.4f}")
print(f"  Mean Confidence: {confidences.mean():.4f}")

In [ ]:
# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    true_labels,
    predictions,
    target_names=label_encoder.classes_,
    digits=4,
))

## Part 10: Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(true_labels, predictions)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    ax=axes[0],
)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    ax=axes[1],
    vmin=0,
    vmax=1,
)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 11: Analyzing Prediction Confidence

In [ ]:
# Confidence distribution by correctness
correct_mask = predictions == true_labels
correct_confidences = confidences[correct_mask]
incorrect_confidences = confidences[~correct_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(
    correct_confidences,
    bins=30,
    alpha=0.7,
    label=f'Correct ({correct_mask.sum()})',
    color='green',
    edgecolor='black',
)
axes[0].hist(
    incorrect_confidences,
    bins=30,
    alpha=0.7,
    label=f'Incorrect ({(~correct_mask).sum()})',
    color='red',
    edgecolor='black',
)
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Confidence Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot by cell type
confidence_df = pd.DataFrame({
    'Cell Type': label_encoder.inverse_transform(true_labels),
    'Confidence': confidences,
    'Correct': correct_mask,
})

sns.boxplot(
    data=confidence_df,
    x='Cell Type',
    y='Confidence',
    ax=axes[1],
)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Confidence by Cell Type')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean confidence (correct): {correct_confidences.mean():.4f}")
print(f"Mean confidence (incorrect): {incorrect_confidences.mean():.4f}")

## Part 12: Predicting on New (Unlabeled) Data

Now you can use your trained model to annotate new cells!

In [ ]:
# Example: Predict on entire dataset
from scgpt_mini.tasks.annotation import annotate_adata

# Annotate (adds predictions to adata.obs)
adata = annotate_adata(
    model=model,
    adata=adata,
    vocab=vocab,
    label_encoder=label_encoder,
    batch_size=64,
    device=device,
    confidence_threshold=0.5,  # Flag low-confidence predictions
)

print("✅ Predictions added to adata.obs['predicted_cell_type']")
print("   Confidence scores in adata.obs['prediction_confidence']")
print("   Low-confidence flags in adata.obs['low_confidence']")

In [ ]:
# Check low-confidence predictions
low_conf_cells = adata.obs[adata.obs['low_confidence']]
print(f"\nLow-confidence predictions: {len(low_conf_cells)} cells")
print(low_conf_cells[['louvain', 'predicted_cell_type', 'prediction_confidence']].head(10))

## Part 13: Saving the Fine-Tuned Model

In [ ]:
# Save fine-tuned model
output_dir = Path('./tutorial_output')
output_dir.mkdir(exist_ok=True)

finetuned_path = output_dir / 'finetuned_annotation_model.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': model_config,
    'label_encoder': label_encoder,
    'vocab': vocab,
    'test_accuracy': accuracy,
    'test_balanced_accuracy': balanced_acc,
}, finetuned_path)

print(f"✅ Fine-tuned model saved to: {finetuned_path}")
print(f"   File size: {finetuned_path.stat().st_size / 1024 / 1024:.2f} MB")

## Summary

In this tutorial, you learned:

✅ **Fine-Tuning Strategies**:
- Frozen encoder (fast, less data)
- Full fine-tuning (better performance, more data)
- When to use each strategy

✅ **Classification Pipeline**:
- Preparing labeled data
- Train/val/test splits
- Training classification models

✅ **Evaluation**:
- Accuracy metrics
- Confusion matrices
- Confidence analysis
- Per-class performance

✅ **Prediction**:
- Annotating new cells
- Confidence thresholding
- Handling uncertainty

✅ **Model Deployment**:
- Saving fine-tuned models
- Loading for inference

## Congratulations!

You've completed all four tutorials and now understand:
1. Data preprocessing and tokenization
2. Model pretraining with MLM
3. Cell embedding generation
4. Cell type annotation

You're ready to apply scGPT-mini to your own single-cell datasets!

## Exercises

Try these to deepen your understanding:

1. **Class imbalance**: Try class weighting in CrossEntropyLoss for imbalanced datasets
2. **Few-shot learning**: Train with only 10 examples per class. How does performance degrade?
3. **Cross-dataset transfer**: Train on PBMC 3k, test on PBMC 10k
4. **Ensemble models**: Train multiple models with different random seeds and ensemble predictions
5. **Active learning**: Identify most uncertain samples for manual annotation
6. **Hierarchical classification**: First classify broad types, then subtypes
7. **Calibration**: Analyze if confidence scores are well-calibrated (reliability diagrams)